# 请求后端 LangGraph Platform 流式接口

本 Notebook 通过 `langgraph-sdk` 直接请求本地 LangGraph Platform 服务（`http://localhost:8123`），与前端 `useAgentChat` 调用的是同一套后端。

- Assistant ID：`paas-agent`
- 流式端点：`/runs/stream`（SDK 自动拼接）
- StreamMode：`values` + `events`

In [ ]:
from langgraph_sdk import get_client

API_URL = "http://localhost:8123"
ASSISTANT_ID = "paas-agent"

client = get_client(url=API_URL)
client

## 1. 检查可用 assistants

In [ ]:
assistants = await client.assistants.search()
for a in assistants:
    print(a["assistant_id"], a["graph_id"], a.get("description"))

## 2. 流式调用 agent（彩色实时打印）

In [ ]:
import json
from datetime import datetime

class Colors:
    BLUE = "\033[94m"
    CYAN = "\033[96m"
    GREEN = "\033[92m"
    YELLOW = "\033[93m"
    BOLD = "\033[1m"
    RESET = "\033[0m"

async def stream_agent_platform(query: str):
    """通过 LangGraph Platform 流式运行 paas-agent，并实时打印事件。"""
    print("=" * 70)
    print(f"{Colors.BOLD}任务: {query}{Colors.RESET}")
    print(f"时间: {datetime.now().strftime('%H:%M:%S')}")
    print("=" * 70)
    print()

    in_thinking = False
    final_answer = ""

    async for part in client.runs.stream(
        thread_id=None,
        assistant_id=ASSISTANT_ID,
        input={
            "messages": [{"type": "human", "content": query}],
            "task_status": "incomplete",
        },
        stream_mode=["values", "events"],
        config={"recursion_limit": 30},
    ):
        # part 是 StreamPart(event=..., data=..., id=...)
        event_name = part.event
        data = part.data

        if event_name == "events":
            inner = data.get("event", "")
            name = data.get("name", "")
            event_data = data.get("data", {})

            if inner == "on_chat_model_stream":
                chunk = event_data.get("chunk", {})
                content = chunk.get("content", "") if isinstance(chunk, dict) else getattr(chunk, "content", "")
                if content:
                    if not in_thinking:
                        print(f"{Colors.BLUE}💭 ", end="")
                        in_thinking = True
                    print(content, end="", flush=True)

            elif inner == "on_tool_start":
                if in_thinking:
                    print()
                    in_thinking = False
                tool_name = event_data.get("tool", name) or "工具"
                print(f"\n{Colors.YELLOW}⚡ 执行工具: {tool_name}{Colors.RESET}")

            elif inner == "on_tool_end":
                output = event_data.get("output", "")
                try:
                    parsed = json.loads(output) if isinstance(output, str) else output
                    display = json.dumps(parsed, ensure_ascii=False)
                except Exception:
                    display = str(output)
                print(f"{Colors.GREEN}✅ 工具返回: {display}{Colors.RESET}\n")

            elif inner == "on_chain_start" and name == "call_model":
                print(f"\n{Colors.BLUE}┌─ 模型思考...{Colors.RESET}")

            elif inner == "on_chain_end" and name == "call_model":
                if in_thinking:
                    print()
                    in_thinking = False
                print(f"{Colors.BLUE}└─ 思考完成{Colors.RESET}\n")

        elif event_name == "values":
            # 每次状态更新时，最后一条 AI 消息就是当前最新回答
            messages = data.get("messages", [])
            if messages:
                last = messages[-1]
                if last.get("type") == "ai" and last.get("content"):
                    final_answer = last["content"]

    print("=" * 70)
    print(f"{Colors.GREEN}✅ 执行完成{Colors.RESET}")
    print(f"最终回答: {final_answer[:200]}{'...' if len(final_answer) > 200 else ''}")
    print("=" * 70)

In [ ]:
await stream_agent_platform("计算 2 + 3")

In [ ]:
await stream_agent_platform("计算 (128 + 256) / 4，然后查询北京天气")

## 3. 原始 HTTP / SSE 请求（不依赖 SDK）

In [ ]:
import requests
import json

url = f"{API_URL}/runs/stream"
payload = {
    "assistant_id": ASSISTANT_ID,
    "input": {
        "messages": [{"type": "human", "content": "计算 23 * 47"}],
        "task_status": "incomplete"
    },
    "stream_mode": ["values", "events"],
    "config": {"recursion_limit": 30}
}

with requests.post(url, json=payload, stream=True) as resp:
    print(f"状态码: {resp.status_code}")
    for line in resp.iter_lines():
        if not line:
            continue
        text = line.decode("utf-8")
        if not text.startswith("data:"):
            continue
        data_json = text[len("data:"):].strip()
        try:
            part = json.loads(data_json)
            # 只打印模型 token，避免输出过长
            if part.get("event") == "events":
                inner = part["data"].get("event", "")
                if inner == "on_chat_model_stream":
                    content = part["data"].get("data", {}).get("chunk", {}).get("content", "")
                    if content:
                        print(content, end="", flush=True)
        except json.JSONDecodeError:
            pass
print()